Data Exploration & Enrichment

In [2]:
import sys
!{sys.executable} -m pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [3]:
import pandas as pd
from pathlib import Path

# Load raw files
df_raw = pd.read_excel("../data/raw/ethiopia_fi_unified_data.xlsx")
df_ref = pd.read_excel("../data/raw/reference_codes.xlsx")

print("Raw Data Shape:", df_raw.shape)
df_raw.head()

Raw Data Shape: (43, 34)


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN


In [4]:
# Check record types distribution
print(df_raw['record_type'].value_counts())

# Check null values and columns
print(df_raw.isnull().sum())

record_type
observation    30
event          10
target          3
Name: count, dtype: int64
record_id               0
record_type             0
category               33
pillar                 10
indicator               0
indicator_code          0
indicator_direction    10
value_numeric          10
value_text             33
value_type              0
unit                   10
observation_date        0
period_start           33
period_end             33
fiscal_year             0
gender                  0
location                0
region                 43
source_name             0
source_type             0
source_url             12
confidence              0
related_indicator      43
relationship_type      43
impact_direction       43
impact_magnitude       43
impact_estimate        43
lag_months             43
evidence_basis         43
comparable_country      0
collected_by            0
collection_date        33
original_text          10
notes                  43
dtype: int64


In [7]:
# Define new enriched observations, events, and impact links following the unified schema
new_records = [
    {
        "record_type": "observation",
        "pillar": "Usage",
        "indicator": "Mobile Internet Penetration Rate",
        "indicator_code": "MOBILE_INTERNET_PENETRATION",
        "value_numeric": 34.5,
        "observation_date": "2024-12-31",
        "source_name": "GSMA Intelligence",
        "source_url": "https://www.gsma.com",
        "confidence": "high",
        "category": "infrastructure",
        "value_type": "percentage",
        "unit": "%",
        "fiscal_year": "2024",
        "gender": "all",
        "location": "national",
        "source_type": "industry_report",
        "comparable_country": "Sub-Saharan Africa Average",
        "collected_by": "Selam Analytics Team",
        "original_text": "Sub-Saharan Africa mobile internet adoption reached..."
    },
    {
        "record_type": "event",
        "pillar": None,
        "indicator": "M-Pesa Ethiopia Commercial Launch",
        "indicator_code": "MARKET_ENTRY_MPESA",
        "observation_date": "2023-08-16",
        "source_name": "Safaricom Ethiopia",
        "source_url": "https://safaricom.et",
        "confidence": "high",
        "category": "product_launch",
        "value_type": "milestone",
        "fiscal_year": "2024",
        "gender": "all",
        "location": "national",
        "source_type": "operator_report",
        "comparable_country": "Kenya",
        "collected_by": "Selam Analytics Team",
        "original_text": "M-Pesa officially launched commercial mobile money services in Ethiopia."
    },
    {
        "record_type": "impact_link",
        "pillar": "Usage",
        "indicator": "M-Pesa Impact Link on Usage",
        "indicator_code": "IMPACT_MPESA_USAGE",
        "related_indicator": "DIGITAL_PAYMENT_ADOPTION",
        "relationship_type": "positive_driver",
        "impact_direction": "positive",
        "impact_magnitude": 5.2,
        "impact_estimate": "medium-high",
        "lag_months": 6,
        "evidence_basis": "comparable market growth trajectory from East Africa",
        "observation_date": "2023-08-16",
        "source_name": "Selam Analytics Research",
        "source_type": "modeled_estimate",
        "confidence": "medium",
        "comparable_country": "Kenya",
        "collected_by": "Selam Analytics Team",
        "notes": "Estimated adoption acceleration post-competitor entry."
    }
]

In [8]:
# Validation Check Before Export
def validate_enriched_dataframe(df):
    required_cols = ["record_type", "indicator_code", "observation_date", "confidence"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Enriched dataset is missing mandatory column: {col}")
    print(f"Validation Passed! Total records in enriched dataset: {len(df)}")
    return True

# Append to dataframe, validate, and export
df_enriched = pd.concat([df_raw, pd.DataFrame(new_records)], ignore_index=True)

if validate_enriched_dataframe(df_enriched):
    output_path = Path("../data/processed/ethiopia_fi_enriched_data.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df_enriched.to_csv(output_path, index=False)
    print(f"Enriched dataset successfully saved to {output_path}")

Validation Passed! Total records in enriched dataset: 46
Enriched dataset successfully saved to ..\data\processed\ethiopia_fi_enriched_data.csv
